### **Integrando Word2Vec**

Este cuaderno tiene como objetivo es reforzar la intuición sobre **representaciones distribuidas**, **CBOW**, **skip-gram** y **similitud semántica** antes de entrar al alineamiento visual-semántico temprano.

> **Alcance:** este material cubre principalmente el lado de **NLP**. La parte de representación visual y alineamiento imagen-texto se trabajará con el material de clase y el cuaderno específico de multimodalidad.

#### **Configuración**

Para este cuaderno, utilizarás las siguientes librerías:

*   [`torch`](https://pandas.pydata.org/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMML0187ENSkillsNetwork31430127-2021-01-01) para construir modelos de redes neuronales y preparar los datos.
*   [`numpy`](https://numpy.org/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMML0187ENSkillsNetwork31430127-2021-01-01) para operaciones matemáticas.
*   [`seaborn`](https://seaborn.pydata.org/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMML0187ENSkillsNetwork31430127-2021-01-01) para visualizar los datos.
*   [`matplotlib`](https://matplotlib.org/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMML0187ENSkillsNetwork31430127-2021-01-01) para herramientas adicionales de graficación.


#### **Instalación opcional de librerías**

En Colab o en un entorno limpio puedes descomentar estas líneas si lo necesitas. Si tu entorno ya tiene PyTorch, torch y scikit-learn, puedes omitir esta sección.

In [ ]:
# Ejecútalo solo si tu entorno no tiene las librerías instaladas.
# %pip install -q numpy pandas matplotlib seaborn scikit-learn torch torchtext torchdata tqdm

In [ ]:
# No es necesario instalar GloVe para este cuaderno.
# Se trabaja con ejemplos pequeños entrenados desde cero usando CBOW y skip-gram.
# Si quisieras explorar embeddings preentrenados más adelante, podrías añadirlos como sección opcional.

#### **Importación de librerías**

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm

%matplotlib inline

import warnings
warnings.filterwarnings("ignore")


Se define una función para graficar los embeddings de palabras en un espacio 2D. La visualización con t-SNE es **solo ilustrativa**: ayuda a observar agrupamientos, pero no debe interpretarse como una evaluación definitiva de la calidad de los embeddings.

In [ ]:
def plot_embeddings(word_embeddings, vocab):
    words = vocab.get_itos()
    if len(words) < 3:
        raise ValueError("Se necesitan al menos 3 palabras para visualizar embeddings.")

    perplexity = min(15, len(words) - 1)
    tsne = TSNE(n_components=2, random_state=0, perplexity=perplexity)
    word_embeddings_2d = tsne.fit_transform(word_embeddings)

    plt.figure(figsize=(15, 15))
    for i, word in enumerate(words):
        plt.scatter(word_embeddings_2d[i, 0], word_embeddings_2d[i, 1])
        plt.annotate(word, (word_embeddings_2d[i, 0], word_embeddings_2d[i, 1]))

    plt.xlabel("Componente t-SNE 1")
    plt.ylabel("Componente t-SNE 2")
    plt.title("Embeddings de palabras visualizados con t-SNE")
    plt.show()


Se define una función que retorna palabras similares a una palabra específica usando **similitud coseno** entre vectores.

In [ ]:
# Esta función retorna las palabras más similares a una palabra objetivo calculando la distancia coseno entre vectores
def find_similar_words(word, word_embeddings, top_k=5):
    if word not in word_embeddings:
        print("Palabra no encontrada en los embeddings.")
        return []

    # Obtiene el embedding para la palabra dada
    target_embedding = word_embeddings[word]

    # Calcula las similitudes coseno entre la palabra objetivo y todas las demás
    similarities = {}
    for w, embedding in word_embeddings.items():
        if w != word:
            similarity = torch.dot(target_embedding, embedding) / (
                torch.norm(target_embedding) * torch.norm(embedding)
            )
            similarities[w] = similarity.item()

    # Ordena las similitudes en orden descendente
    sorted_similarities = sorted(similarities.items(), key=lambda x: x[1], reverse=True)

    # Retorna las top k palabras similares
    most_similar_words = [w for w, _ in sorted_similarities[:top_k]]
    return most_similar_words


Se define una función para entrenar un modelo sencillo de embeddings con datos sintéticos. Para mantener el cuaderno ligero, el número de épocas se dejará en un valor moderado.

In [ ]:
def train_model(modelo, dataloader, criterion, optimizer, num_epochs=1000):
    """
    Entrena el modelo durante el número especificado de épocas.
    
    Args:
        modelo: El modelo de PyTorch a entrenar.
        dataloader: DataLoader que proporciona los datos para entrenamiento.
        criterion: Función de pérdida.
        optimizer: Optimizador para actualizar los pesos del modelo.
        num_epochs: Número de épocas para entrenar el modelo.

    Returns:
        modelo: El modelo entrenado.
        epoch_losses: Lista de pérdidas promedio para cada época.
    """
    
    # Lista para almacenar la pérdida en cada época
    epoch_losses = []

    for epoch in tqdm(range(num_epochs)):
        # Almacena la pérdida acumulada para la época actual
        running_loss = 0.0

        # Uso de tqdm para una barra de progreso
        for idx, samples in enumerate(dataloader):

            optimizer.zero_grad()
            
            # Comprueba si el modelo contiene una capa EmbeddingBag
            if any(isinstance(module, nn.EmbeddingBag) for _, module in modelo.named_modules()):
                target, context, offsets = samples
                predicted = modelo(context, offsets)
            
            # Comprueba si el modelo contiene una capa Embedding
            elif any(isinstance(module, nn.Embedding) for _, module in modelo.named_modules()):
                target, context = samples
                predicted = modelo(context)
                
            loss = criterion(predicted, target)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(modelo.parameters(), 0.1)
            optimizer.step()
            running_loss += loss.item()

        # Agrega la pérdida promedio de la época
        epoch_losses.append(running_loss / len(dataloader))
    
    return modelo, epoch_losses


#### **Word2Vec**

Word2Vec es una familia de métodos que transforma palabras en vectores numéricos, posicionando palabras similares cerca unas de otras en un espacio vectorial. De esta manera, es posible cuantificar y analizar relaciones semánticas de forma matemática. Por ejemplo, palabras como *cat* y *kitten* o *cat* y *dog* tienden a quedar cercanas, mientras que una palabra como *book* suele quedar más alejada.

<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-AI0205EN-SkillsNetwork/Words.png" alt="Ejemplo de Word2Vec" class="bg-primary" width="400px">

#### **Crear y entrenar modelos Word2Vec**

In [ ]:
toy_data = """I wish I was a baller
I wish I was taller
The dog chased the ball
The puppy chased the toy
The cat played with the kitten
The small child took careful steps
The little toddler learned to walk
The big house had a large garden
The huge building stood tall
The chef cooked a delicious meal
The artist painted a beautiful picture
The team won the championship"""

A continuación, se prepara el texto, se tokeniza y se construye un vocabulario a partir de todos los tokens del corpus de juguete.

In [ ]:
def basic_tokenizer(text):
    # tokenización sencilla: minúsculas y separación de palabras/ signos básicos
    return re.findall(r"[a-zA-Z']+|[.,!?;]", text.lower())

class SimpleVocab:
    def __init__(self, tokens, specials=None):
        specials = specials or []
        unique_tokens = []
        seen = set()
        for token in specials + list(tokens):
            if token not in seen:
                seen.add(token)
                unique_tokens.append(token)
        self.itos = unique_tokens
        self.stoi = {tok: i for i, tok in enumerate(self.itos)}
        self.default_index = self.stoi.get("<unk>", 0)

    def __getitem__(self, token):
        return self.stoi.get(token, self.default_index)

    def get_itos(self):
        return self.itos

    def set_default_index(self, idx):
        self.default_index = idx

    def __len__(self):
        return len(self.itos)

# Paso 1: tokeniza el corpus completo
tokenized_toy_data = basic_tokenizer(toy_data)

# Paso 2: construye el vocabulario
vocab = SimpleVocab(tokenized_toy_data, specials=["<unk>"])
vocab.set_default_index(vocab["<unk>"])

print(f"Número de tokens en el corpus: {len(tokenized_toy_data)}")
print(f"Tamaño del vocabulario: {len(vocab)}")
print("Primeros 15 tokens:", tokenized_toy_data[:15])


Veamos cómo se ve una secuencia después de la tokenización y de la numeración:

In [ ]:
# Prueba
sample_sentence = "I wish I was  a baller"
tokenized_sample = basic_tokenizer(sample_sentence)
encoded_sample = [vocab[token] for token in tokenized_sample]
print("Muestra codificada:", encoded_sample)

Se escribe una función para convertir tokens a índices del vocabulario:

In [ ]:
text_pipeline = lambda tokens: [vocab[token] for token in tokens]

#### **Bolsa continua de palabras (CBOW)**

<table border="1">
    <tr>
        <th>Paso de tiempo</th>
        <th>Frase</th>
    </tr>
    <tr>
        <td>1</td>
        <td><span style="color:blue;">I wish</span> <span style="color:red;">I</span> <span style="color:blue;">was  little </span></td>
    </tr>
    <tr>
        <td>2</td>
        <td><span style="color:blue;">wish I</span> <span style="color:red;">was</span> <span style="color:blue;">little bit </span></td>
    </tr>
    <tr>
        <td>3</td>
        <td><span style="color:blue;">I was</span> <span style="color:red;">little</span> <span style="color:blue;">  bit taller</span></td>
    </tr>
    <tr>
        <td>4</td>
        <td><span style="color:blue;">was little</span> <span style="color:red;">bit</span> <span style="color:blue;"> taller I</span></td>
    </tr>
    <tr>
        <td>5</td>
        <td><span style="color:blue;">little bit</span> <span style="color:red;">taller</span> <span style="color:blue;"> I wish</span></td>
    </tr>
    <tr>
        <td>6</td>
        <td><span style="color:blue;">bit taller</span> <span style="color:red;">I</span> <span style="color:blue;">wish I</span></td>
    </tr>
    <tr>
        <td>7</td>
        <td><span style="color:blue;">taller I</span> <span style="color:red;">wish</span> <span style="color:blue;">I was</span></td>
    </tr>
    <tr>
        <td>8</td>
        <td><span style="color:blue;">I wish</span> <span style="color:red;">I</span> <span style="color:blue;">was a</span></td>
    </tr>
    <tr>
        <td>9</td>
        <td><span style="color:blue;">wish I</span> <span style="color:red;">was</span> <span style="color:blue;">a baller</span></td>
    </tr>
</table>



Se desplaza una ventana de contexto sobre la secuencia para crear los ejemplos de entrenamiento del modelo **CBOW**.

In [ ]:
CONTEXT_SIZE = 2

cbow_data = []
for i in range(CONTEXT_SIZE, len(tokenized_toy_data) - CONTEXT_SIZE):
    context = (
        [tokenized_toy_data[i - j - 1] for j in range(CONTEXT_SIZE)]
        + [tokenized_toy_data[i + j + 1] for j in range(CONTEXT_SIZE)]
    )
    target = tokenized_toy_data[i]
    cbow_data.append((context, target))


Imprimimos un ejemplo, mostrando tanto las palabras de contexto como la palabra objetivo.

In [ ]:
print(cbow_data[1])

Se imprime otra muestra para comprobar cómo cambia la ventana de contexto.

In [ ]:
print(cbow_data[2])

Se aprenden embeddings que ayuden al modelo a predecir la palabra objetivo a partir del contexto. En términos probabilísticos, el modelo aproxima $P(w_t\mid w_{t-2},w_{t-1},w_{t+1},w_{t+2})$.

<table border="1">
    <thead>
        <tr>
            <th>$ P(w_t| w_{t-2},w_{t-1},w_{t+1},w_{t+2})$</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td>$P(w_1 | w_{-1},w_0,w_2,w_3) = P(I | \text{I wish, was little}) $</td>
        </tr>
        <tr>
            <td>$P(w_2 | w_0,w_1,w_3,w_4) = P(was | \text{wish I, little bit}) $</td>
        </tr>
        <tr>
            <td>$P(w_3 | w_1,w_2,w_4,w_5) = P(little | \text{I was, bit taller}) $</td>
        </tr>
        <tr>
            <td>$P(w_4 | w_2,w_3,w_5,w_6) = P(bit | \text{was little, taller I}) $</td>
        </tr>
        <tr>
            <td>$P(w_5 | w_3,w_4,w_6,w_7) = P(taller | \text{little bit, I wish}) $</td>
        </tr>
        <tr>
            <td>$P(w_6 | w_4,w_5,w_7,w_8) = P(I | \text{bit taller, wish I}) $</td>
        </tr>
        <tr>
            <td>$P(w_7 | w_5,w_6,w_8,w_9) = P(wish | \text{taller I, I was}) $</td>
        </tr>
    </tbody>
</table>


La función `collate_batch` procesa lotes de datos, convirtiendo el texto (contexto y palabra objetivo) a formato numérico usando el vocabulario y organizándolos para el entrenamiento del modelo.


In [ ]:
def collate_batch(batch):
    target_list, context_list, offsets = [], [], [0]
    for _context, _target in batch:
        
        target_list.append(vocab[_target])  
        processed_context = torch.tensor(text_pipeline(_context), dtype=torch.int64)
        context_list.append(processed_context)
        offsets.append(processed_context.size(0))
    target_list = torch.tensor(target_list, dtype=torch.int64)
    offsets = torch.tensor(offsets[:-1]).cumsum(dim=0)
    context_list = torch.cat(context_list)
    return target_list.to(device), context_list.to(device), offsets.to(device)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


Se seleccionan las primeras 10 muestras de `cbow_data` y se procesan usando la función `collate_batch`. Los resultados muestran índices de palabras, contexto concatenado y offsets.

In [ ]:
target_list, context_list, offsets = collate_batch(cbow_data[0:10])
print(target_list[:5], context_list[:10], offsets[:5])

Se crea un objeto `DataLoader` para alimentar el entrenamiento en lotes.

In [ ]:
BATCH_SIZE = 64

dataloader_cbow = DataLoader(cbow_data, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch)

El modelo CBOW que se muestra a continuación comienza con una capa `EmbeddingBag`, que toma una lista de índices de palabras de contexto (de longitud variable) y produce el promedio de los embeddings de dichas palabras. Este embedding se pasa luego a través de una capa lineal que reduce su dimensión a `embed_dim/2`. 

Tras aplicar la activación ReLU, la salida se procesa con otra capa lineal, transformándola para que coincida con el tamaño del vocabulario, permitiendo así que el modelo prediga la probabilidad de cualquier palabra del vocabulario como palabra objetivo. 
El flujo general va desde los índices de las palabras de contexto hasta predecir la palabra central en el enfoque de CBOW.


In [ ]:
class CBOW(nn.Module):
    # Inicializa el modelo CBOW
    def __init__(self, vocab_size, embed_dim, num_class):
        
        super(CBOW, self).__init__()
         # Define la capa de embedding usando nn.EmbeddingBag
        # Esta capa produce el promedio de los embeddings de las palabras de contexto
        self.embedding = nn.EmbeddingBag(vocab_size, embed_dim, sparse=False)
        # Define la primera capa lineal con tamaño de entrada embed_dim y tamaño de salida embed_dim//2
        self.linear1 = nn.Linear(embed_dim, embed_dim//2)
        # Define la capa completamente conectada con tamaño de entrada embed_dim//2 y tamaño de salida vocab_size
        self.fc = nn.Linear(embed_dim//2, vocab_size)
        
        self.init_weights()
    # Inicializa los pesos de los parámetros del modelo
    def init_weights(self):
        # Inicializa los pesos de la capa de embedding
        initrange = 0.5
        self.embedding.weight.data.uniform_(-initrange, initrange)
        # Inicializa los pesos de la capa completamente conectada
        self.fc.weight.data.uniform_(-initrange, initrange)
        # Inicializa los sesgos de la capa completamente conectada a ceros
        self.fc.bias.data.zero_()
        
    def forward(self, texto, offsets):
        # Pasa el texto de entrada y los offsets a través de la capa de embedding
        out = self.embedding(texto, offsets)
        # Aplica la función de activación ReLU a la salida de la primera capa lineal
        out = torch.relu(self.linear1(out))
        # Pasa la salida de la activación ReLU a través de la capa completamente conectada
        return self.fc(out)


Se crea una instancia del modelo CBOW:


In [ ]:
vocab_size = len(vocab)
emsize = 24
model_cbow = CBOW(vocab_size, emsize, vocab_size).to(device)


Se define la función de pérdida y el optimizador para el entrenamiento.

In [ ]:
LR = 5  # tasa de aprendizaje

# Define el criterio CrossEntropyLoss. Es comúnmente usado para tareas de clasificación multiclase.
# Este criterio combina la función softmax con la pérdida de log-verosimilitud negativa.
criterion = torch.nn.CrossEntropyLoss()

# Define el optimizador utilizando descenso de gradiente estocástico (SGD).
# Optimiza los parámetros del modelo model_cbow, obtenidos con model_cbow.parameters().
# La tasa de aprendizaje (lr) determina el tamaño del paso en cada actualización de los parámetros.
optimizer = torch.optim.SGD(model_cbow.parameters(), lr=LR)

# Define un scheduler para la tasa de aprendizaje.
# El scheduler StepLR ajusta la tasa de aprendizaje durante el entrenamiento.
# Multiplica la tasa de aprendizaje por gamma cada 1 época (aquí, 1.0).
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, 1.0, gamma=0.1)


Se entrena el modelo. El número de épocas se reduce para que el cuaderno siga siendo ligero y útil como material de repaso.

In [ ]:
model_cbow, epoch_losses = train_model(model_cbow, dataloader_cbow, criterion, optimizer, num_epochs=200)

Graficamos los valores de pérdida a lo largo del entrenamiento:


In [ ]:
plt.plot(epoch_losses)
plt.xlabel("épocas")


Los pesos del modelo son los embeddings de palabras reales. Se pueden cargar en un arreglo de numpy:


In [ ]:
word_embeddings = model_cbow.embedding.weight.detach().cpu().numpy()


Revisamos el vector embebido para una palabra de ejemplo. Observamos la dimensión de este vector, que es igual a `emsize = 24` que se definió anteriormente.


In [ ]:
word = 'baller'
word_index = vocab.stoi[word]  # obtener el índice de la palabra en el vocabulario
print(word_embeddings[word_index])


Podemos comprobar si los embeddings capturan similitudes entre palabras. Para visualizar mejor el espacio, reducimos la dimensionalidad con t-SNE.

In [ ]:
plot_embeddings(word_embeddings, vocab=vocab)

Al examinar las proyecciones de t-SNE, se evidencia que, pese a la inevitable pérdida de información por la reducción de dimensiones y las limitaciones de un conjunto de datos pequeño, las palabras con significados similares se agrupan. 

Por ejemplo, palabras como 'bright' y 'shadow' se encuentran próximas cerca del punto $(-15, -15)$ en los componentes 1 y 2. De igual forma, 'cat', 'dog' y 'mouse' se agrupan alrededor del punto $(5, 5)$, y 'sailed' junto con 'wind' pueden encontrarse próximos al punto $(5, -8)$.


#### **Modelo skip-gram**

<table border="1">
    <tr>
        <th>Contexto completo con objetivo</th>
        <th>Palabra objetivo</th>
        <th>Contexto original del objetivo</th>
        <th>Approximación 1</th>
        <th>Approximación 2</th>
        <th>Approximación 3</th>
        <th>Approximación 4</th>
    </tr>
    <tr>
        <td><span style="color:red;"> I was</span> <span style="color:blue;">little</span> <span style="color:red;">bit taller, </span></td>
        <td>little</td>
        <td> I was bit taller,</td>
        <td>I</td>
        <td>was</td>
        <td>bit</td>
        <td>taller,</td>
    </tr>
    <tr>
        <td><span style="color:red;"> was little</span> <span style="color:blue;">bit</span> <span style="color:red;">taller, I </span></td>
        <td>bit</td>
        <td>was little taller, I</td>
        <td>was</td>
        <td>little</td>
        <td>taller,</td>
        <td>I</td>
    </tr>
    <tr>
        <td><span style="color:red;">little bit</span> <span style="color:blue;">taller,</span> <span style="color:red;">I wish </span></td>
        <td>taller,</td>
        <td>little bit I wish</td>
        <td>little</td>
        <td>bit</td>
        <td>I</td>
        <td>wish</td>
    </tr>
    <tr>
        <td><span style="color:red;"> bit taller,</span> <span style="color:blue;">I</span> <span style="color:red;">wish I </span></td>
        <td>I</td>
        <td>bit taller, wish I</td>
        <td>bit</td>
        <td>taller,</td>
        <td>wish</td>
        <td>I</td>
    </tr>
    <tr>
        <td><span style="color:red;"> taller, I</span> <span style="color:blue;">wish</span> <span style="color:red;">I was </span></td>
        <td>wish</td>
        <td>taller, I I was</td>
        <td>taller,</td>
        <td>I</td>
        <td>I</td>
        <td>was</td>
    </tr>
    <!-- More rows can be added in a similar pattern for other words in the phrase. -->
</table>


El objetivo es optimizar las probabilidades condicionales para obtener embeddings, optimizar las probabilidades condicionales para obtener embeddings de alta calidad. 

> La única diferencia con respecto al CBOW estándar es la estructura de las probabilidades condicionales $P(w_{t+j}| w_{t})$ para una ventana de $j=-2,-1,..,1,2$.


<table border="1">
    <tr>
        <th>j</th>
        <th>Palabra objetivo t=3 </th>
        <th>Palabra de contexto</th>
        <th>Probabilidad</th>
    </tr>
    <tr>
         <th>-1</th>
        <td>little</td>
        <td>was</td>
        <td> P(was | little)</td>
    </tr>
    <tr>
         <th>1</th>
        <td>little</td>
        <td>bit</td>
        <td>P(bit | little)</td>
    </tr>
    <tr>
         <th>2</th>
        <td>little</td>
        <td>taller</td>
        <td>P(taller | little) </td>
    </tr>
    <!-- Repeat rows for each context word for each target word -->
</table>


En contraste con la notación estándar en probabilidad condicional, donde la variable dependiente se representa generalmente como la objetivo, la terminología actual invierte esta convención.


Este código construye un conjunto de datos para el modelo skip-gram a partir de datos tokenizados, en el que para cada palabra (objetivo) se recogen las palabras circundantes dentro de una ventana especificada (contexto) definida por `CONTEXT_SIZE`.


In [ ]:
# Define el tamaño de la ventana para el contexto alrededor de la palabra objetivo.
CONTEXT_SIZE = 2

# Inicializa una lista vacía para almacenar los pares (objetivo, contexto).
skip_data = []

# Itera sobre cada palabra en tokenized_toy_data, excluyendo las primeras y últimas palabras según CONTEXT_SIZE.
for i in range(CONTEXT_SIZE, len(tokenized_toy_data) - CONTEXT_SIZE):

    # Para la palabra en la posición i, el contexto comprende las palabras anteriores (de acuerdo al CONTEXT_SIZE)
    # y las palabras siguientes. Se recoge el contexto en una lista.
    context = (
        [tokenized_toy_data[i - j - 1] for j in range(CONTEXT_SIZE)]  # Palabras precedentes
        + [tokenized_toy_data[i + j + 1] for j in range(CONTEXT_SIZE)]  # Palabras siguientes
    )

    # La palabra en la posición i se toma como objetivo.
    target = tokenized_toy_data[i]

    # Agrega el par (objetivo, contexto) a la lista skip_data.
    skip_data.append((target, context))


Se aplica una ventana de contexto al modelo **skip-gram** para construir pares de entrenamiento.

In [ ]:
skip_data_ = [[(sample[0], word) for word in sample[1]] for sample in skip_data]

Luego se aplanan los pares `(objetivo, contexto)` para obtener el conjunto final de ejemplos.

In [ ]:
skip_data_flat = [item for items in skip_data_ for item in items]
skip_data_flat[8:28]


Se crea una función `collate_fn` para convertir los pares `(objetivo, contexto)` en tensores numéricos.

In [ ]:
def collate_fn(batch):
    target_list, context_list = [], []
    for _context, _target in batch:
        
        target_list.append(vocab[_target]) 
        context_list.append(vocab[_context])
        
    target_list = torch.tensor(target_list, dtype=torch.int64)
    context_list = torch.tensor(context_list, dtype=torch.int64)
    return target_list.to(device), context_list.to(device)


In [ ]:
dataloader = DataLoader(skip_data_flat, batch_size=BATCH_SIZE, collate_fn=collate_fn)

Se revisa una muestra de lote del par `(objetivo, contexto)` después de la colación:

In [ ]:
next(iter(dataloader))


Aquí se define la red **skip-gram**.

La capa de embeddings se implementa con `nn.Embedding`, que aprende un vector para cada palabra del vocabulario. La capa `fc` proyecta ese embedding al tamaño del vocabulario para predecir la palabra de contexto.

In [ ]:
class SkipGram_Model(nn.Module):

    def __init__(self, vocab_size, embed_dim):
        super(SkipGram_Model, self).__init__()
        # Define la capa de embeddings
        self.embeddings = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim
        )
        
        # Define la capa completamente conectada
        self.fc = nn.Linear(in_features=embed_dim, out_features=vocab_size)

    def forward(self, texto):
        # Realiza el pase forward
        # Pasa el texto de entrada por la capa de embeddings
        out = self.embeddings(texto)
        
        # Pasa la salida de la capa de embeddings por la capa fc
        # Aplica la función de activación ReLU
        out = torch.relu(out)
        out = self.fc(out)
        
        return out


Se crea una instancia del modelo:


In [ ]:
emsize = 24
model_sg = SkipGram_Model(vocab_size, emsize).to(device)


Se entrena el modelo con datos simples:

In [ ]:
LR = 5  # tasa de aprendizaje
#BATCH_SIZE = 64  # tamaño del lote para el entrenamiento

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model_sg.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, 1.0, gamma=0.1)


In [ ]:
model_sg, epoch_losses = train_model(model_sg, dataloader, criterion, optimizer, num_epochs=200)

In [ ]:
plt.plot(epoch_losses)


También se pueden graficar los embeddings del modelo skip-gram reduciendo sus dimensiones con t-SNE.

In [ ]:
word_embeddings = model_sg.embeddings.weight.detach().cpu().numpy()
plot_embeddings(word_embeddings, vocab=vocab)

#### **Cierre**

En este cuaderno repasaste los conceptos textuales más importantes como:
- representación distribuida de palabras,
- aprendizaje de embeddings con **CBOW** y **skip-gram**,
- similitud coseno entre vectores,
- visualización e interpretación básica de embeddings.

Estos conceptos son la base para comprender después cómo se construyen espacios compartidos **imagen-texto** en los modelos tempranos de alineamiento visual-semántico.

> **Importante:** este cuaderno cubre solo el componente textual. La representación visual, BOVW, CNN features y el alineamiento imagen-texto se revisarán en el material específico de multimodalidad.